# Load WISDM Data into Spark

## 1. Start Spark Session

In [3]:
import os
os.environ["JAVA_HOME"] = "/Library/Java/JavaVirtualMachines/jdk-18.0.1.1.jdk/Contents/Home"

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, LongType, FloatType
from pyspark.sql.functions import regexp_replace, col

spark = SparkSession.builder \
    .appName("WISDM Activity Classifier") \
    .master("local[*]") \
    .getOrCreate()

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/12 23:54:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 2. Load Data

In [4]:
schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("activity", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("x", FloatType(), True),
    StructField("y", FloatType(), True),
    StructField("z", FloatType(), True),
])

data_path = os.path.join(os.getcwd(), "data/wisdm-dataset/raw/phone/accel/*.txt")

raw_df = spark.read \
    .option("sep", ",") \
    .option("header", False) \
    .schema(schema) \
    .csv(data_path)

df = raw_df.withColumn("z", regexp_replace(col("z").cast("string"), ";", "").cast(FloatType()))

df.printSchema()
df.show(5)

26/05/12 23:54:52 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /Users/nataliavillalobos/activity_classifier/data/wisdm-dataset/raw/phone/accel/*.txt.
java.io.FileNotFoundException: File /Users/nataliavillalobos/activity_classifier/data/wisdm-dataset/raw/phone/accel/*.txt does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$ca

root
 |-- user_id: integer (nullable = true)
 |-- activity: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- x: float (nullable = true)
 |-- y: float (nullable = true)
 |-- z: float (nullable = true)

+-------+--------+----------------+-----------+-----------+----+
|user_id|activity|       timestamp|          x|          y|   z|
+-------+--------+----------------+-----------+-----------+----+
|   1646|       A|1467898544326000| -1.8968556| -3.0225148|NULL|
|   1646|       A|1467898564454000|-0.35685793| -3.9206471|NULL|
|   1646|       A|1467898584614000|  2.5698562| -4.8834453|NULL|
|   1646|       A|1467898604806000|-0.17244141| -7.0174074|NULL|
|   1646|       A|1467898624966000|  -6.100115|-11.1176815|NULL|
+-------+--------+----------------+-----------+-----------+----+
only showing top 5 rows


In [5]:
print(f"Total rows: {df.count():,}")
df.groupBy("activity").count().orderBy("activity").show()

Total rows: 4,804,403


+--------+------+
|activity| count|
+--------+------+
|       A|279817|
|       B|268409|
|       C|255645|
|       D|264592|
|       E|269604|
|       F|246356|
|       G|269609|
|       H|270756|
|       I|261360|
|       J|249793|
|       K|285190|
|       L|265781|
|       M|278766|
|       O|272219|
|       P|272730|
|       Q|260497|
|       R|268065|
|       S|265214|
+--------+------+

